# 02 — Transcriber

**Module notebook — definitions only.**

Produces the RAW transcript only — exactly what was said, auto-detected
language/script, never forced. Translation into a chosen output language is
now a separate, explicit step the caller controls (see `translate_transcript()`
below and `content_agent` in `09_agents.ipynb`), so **audio type**,
**transcript language**, and **summary language** can all be chosen
independently instead of one setting doing three jobs.

Depends on: `os`, `get_llm()`, `output_language_instruction()`,
`TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT` (loaded in `00_llm_config.ipynb`).


In [ ]:
import os
import torch
import whisper
import numpy as np
from pydub import AudioSegment
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use the GPU automatically if PyTorch can see one (falls back to CPU
# otherwise, so this is safe to run unchanged on a CPU-only machine).
WHISPER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WHISPER_DTYPE = torch.bfloat16 if WHISPER_DEVICE == "cuda" else torch.float32

WHISPER_MODEL = os.getenv("WHISPER_MODEL", "small")

# Comma-separated list of proper nouns / jargon Whisper commonly mishears
# (product names, brand names, domain terms). Passed as an `initial_prompt`
# to bias its predictions toward the correct spelling for terms it rarely
# saw in training — e.g. without this, "Claude Sonnet" has been observed
# coming out as "Claude saw it", and "PostgreSQL" as "poseGray". Override
# via the WHISPER_VOCABULARY_HINT env var for content-specific vocabulary;
# this default list is general AI/tech terminology.
WHISPER_VOCABULARY_HINT = os.getenv(
    "WHISPER_VOCABULARY_HINT",
    "Claude, GPT, OpenAI, Hugging Face, LangChain, LangGraph, PostgreSQL, "
    "Kubernetes, Docker, Redis, MCP, RAG, vector database, embedding, "
    "semantic chunking, PyTorch, AWS, GCP.",
)

# Whisper hallucination-suppression settings. Without these, one bad guess
# can "poison" every following segment (Whisper conditions each segment on
# its own prior output by default), and low-confidence audio (intro music,
# silence, unclear speech) can produce plausible-sounding but wrong text
# instead of being flagged as low-confidence.
WHISPER_CONDITION_ON_PREVIOUS_TEXT = False
WHISPER_NO_SPEECH_THRESHOLD = 0.6
WHISPER_LOGPROB_THRESHOLD = -1.0

# Whether fp16 is safe on this GPU. Defaults to False (force fp32) since
# fp16 has been observed to produce NaN logits / empty transcripts on some
# GPUs (see the fp16= argument in transcribe_chunk_whisper_raw below).
# Override by setting WHISPER_FP16_SAFE=true in .env once you've confirmed
# fp16 works fine on your specific GPU, for a speed boost.
WHISPER_FP16_SAFE = os.getenv("WHISPER_FP16_SAFE", "false").strip().lower() == "true"

# "hinglish" (an audio_type, not an output language) routes to a Whisper
# large-v3 checkpoint fine-tuned specifically for Hindi-English code-switched
# speech — benchmarked by its creators as competitive with Sarvam's Saaras-v3
# on real Hinglish audio. Runs locally via `transformers`, downloaded once
# from Hugging Face — no API key required.
HINGLISH_ASR_MODEL_ID = os.getenv("HINGLISH_ASR_MODEL_ID", "Trelis/whisper-hinglish-preview")

# WhisperFeatureExtractor (used by the `transformers` Hinglish model below)
# pads/truncates ANY single input to exactly this many seconds (3000 mel
# frames) — feed it more than this in one call and everything past this
# point is silently dropped, with no error or warning. `openai-whisper`
# (used for the auto-detect path below) does its own internal sliding-window
# chunking and isn't affected by this limit, so this constant only matters
# for the Hinglish path.
HINGLISH_ASR_CHUNK_SECONDS = 30
HINGLISH_SAMPLE_RATE = 16000

# Prompt used to turn the raw (possibly non-target-language or mixed-script)
# transcript into a chosen output language. Kept generic/reusable — the
# actual language rule comes from output_language_instruction()
# (00_llm_config.ipynb).
TRANSCRIPT_TRANSLATE_PROMPT = (
    "The text below is a raw, word-for-word transcript of spoken audio. It "
    "may be in a different language than requested, or contain a mix of "
    "scripts/languages (code-switching). Rewrite it as a complete, faithful "
    "transcript in the required output language — translate where needed, "
    "but do NOT summarize, shorten, omit, or add anything: every sentence "
    "and detail from the original must be preserved, in the same order. "
    "Only output the rewritten transcript text, nothing else (no preamble, "
    "no notes)."
)

_model = None
_hinglish_processor = None
_hinglish_asr_model = None


In [ ]:
def load_model():
    """Lazily load and cache the local Whisper model."""
    global _model
    if _model is None:
        print(f"Loading Whisper model: {WHISPER_MODEL} (device={WHISPER_DEVICE}) ...")
        _model = whisper.load_model(WHISPER_MODEL, device=WHISPER_DEVICE)
        print("Whisper model loaded.")
    return _model


def transcribe_chunk_whisper_raw(chunk_path: str) -> str:
    """Transcribe with Whisper using auto language detection (language=None).

    Deliberately never forces a language: forcing a language on audio that
    doesn't match it produces garbled/hallucinated output (this is what
    caused non-English garbage at the start/end of some transcripts before),
    whereas auto-detect transcribes whatever was actually said.

    Also applies WHISPER_VOCABULARY_HINT (fixes mangled proper nouns/jargon,
    e.g. "Claude Sonnet" -> "Claude saw it") and hallucination-suppression
    settings (prevents one bad guess from cascading into surrounding text,
    and suppresses low-confidence garbage during intro music/silence)."""
    model = load_model()
    result = model.transcribe(
        chunk_path,
        task="transcribe",
        language=None,
        initial_prompt=WHISPER_VOCABULARY_HINT,
        condition_on_previous_text=WHISPER_CONDITION_ON_PREVIOUS_TEXT,
        no_speech_threshold=WHISPER_NO_SPEECH_THRESHOLD,
        logprob_threshold=WHISPER_LOGPROB_THRESHOLD,
        # Fix: openai-whisper defaults to fp16=True on any CUDA device. Several
        # GPUs (smaller/older/laptop chips - e.g. the Turing-class T1200 this
        # was hit on) produce NaN logits partway through decoding under fp16,
        # which surfaces as "Expected parameter logits ... found invalid
        # values: tensor([[nan, nan, ...]])" and an empty transcript. Forcing
        # fp32 fixes it - slower than fp16, but still much faster than CPU,
        # and correctness beats the extra speed here.
        fp16=(WHISPER_DEVICE == "cuda" and WHISPER_FP16_SAFE),
    )
    return result["text"]


def load_hinglish_asr_model():
    """Lazily load and cache the Hinglish-specialized ASR model (separate
    checkpoint from the general Whisper model above, loaded once)."""
    global _hinglish_processor, _hinglish_asr_model
    if _hinglish_asr_model is None:
        print(f"Loading Hinglish ASR model: {HINGLISH_ASR_MODEL_ID} (device={WHISPER_DEVICE}) ...")
        _hinglish_processor = WhisperProcessor.from_pretrained(HINGLISH_ASR_MODEL_ID)
        _hinglish_asr_model = (
            WhisperForConditionalGeneration.from_pretrained(
                HINGLISH_ASR_MODEL_ID, torch_dtype=WHISPER_DTYPE
            )
            .to(WHISPER_DEVICE)
            .eval()
        )
        print("Hinglish ASR model loaded.")
    return _hinglish_processor, _hinglish_asr_model


def _load_audio_as_16k_mono(chunk_path: str) -> np.ndarray:
    """Load a chunk with pydub and return 16kHz mono float32 samples in
    [-1, 1] — the format the ASR model's feature extractor expects. Chunks
    from a YouTube download aren't guaranteed to already be 16kHz mono
    (only convert_to_wav() forces that for local-file uploads), so we
    normalize here regardless of source."""
    audio = AudioSegment.from_wav(chunk_path).set_channels(1).set_frame_rate(HINGLISH_SAMPLE_RATE)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32)
    return samples / (2 ** (8 * audio.sample_width - 1))


def transcribe_chunk_hinglish_raw(chunk_path: str) -> str:
    """Transcribe Hindi/English code-switched speech with a model fine-tuned
    for exactly this, using its dedicated <|mixedcode|> token as documented
    on the model card. Returns the RAW mixed-script text (Devanagari for
    Hindi words, Latin for English words mixed in) — no translation here.

    Loops over the chunk in HINGLISH_ASR_CHUNK_SECONDS windows and
    concatenates each window's text, since WhisperFeatureExtractor always
    pads/truncates a single call to that many seconds — without this loop,
    everything past the first window is silently dropped."""
    processor, model = load_hinglish_asr_model()
    audio = _load_audio_as_16k_mono(chunk_path)

    window_samples = HINGLISH_ASR_CHUNK_SECONDS * HINGLISH_SAMPLE_RATE
    ids = processor.tokenizer.convert_tokens_to_ids
    mixedcode_ids = processor.tokenizer("<|mixedcode|>", add_special_tokens=False).input_ids
    prompt = [
        ids("<|startoftranscript|>"),
        ids("<|hi|>"),  # Hindi as the dominant script for Hinglish audio
        *mixedcode_ids,
        ids("<|transcribe|>"),
        ids("<|notimestamps|>"),
    ]

    raw_parts = []
    total_windows = max(1, -(-len(audio) // window_samples))  # ceil division, for logging only
    for window_index, start in enumerate(range(0, len(audio), window_samples), start=1):
        window = audio[start:start + window_samples]
        if window.size == 0:
            continue

        features = processor.feature_extractor(
            window, sampling_rate=HINGLISH_SAMPLE_RATE, return_tensors="pt"
        ).input_features.to(WHISPER_DEVICE, WHISPER_DTYPE)

        with torch.no_grad():
            output_ids = model.generate(
                input_features=features,
                decoder_input_ids=torch.tensor([prompt]).to(WHISPER_DEVICE),
                max_new_tokens=440,
            )
        window_text = processor.tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        if window_text:
            raw_parts.append(window_text)
        print(f"  Hinglish ASR window {window_index}/{total_windows} done.")

    return " ".join(raw_parts)


## Translating the raw transcript

Mirrors the single-call / chunked-fallback pattern used by `summarize()` in
`03_summarizer.ipynb`: almost every real transcript fits in one LLM call
(`TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT`, from `00_llm_config.ipynb`), so that's
the default path. Unlike summarization, translation is 1:1 — chunks are
translated independently and concatenated in order, not combined/reduced,
since there's no cross-chunk synthesis needed (or wanted: doing so would risk
paraphrasing content out of the transcript).

Called explicitly now (not automatically inside transcription), so the
**transcript language** can be chosen independently of **audio type** and
**summary language** — see `content_agent` in `09_agents.ipynb`.

In [ ]:
def _translate_transcript_single_call(raw_transcript: str, language: str) -> str:
    llm = get_llm()
    lang_instruction = output_language_instruction(language)
    prompt = f"{TRANSCRIPT_TRANSLATE_PROMPT}\n\n{lang_instruction}\n\nTranscript:\n{raw_transcript}"
    response = llm.invoke(prompt)
    return response.content.strip()


def _translate_transcript_chunked(raw_transcript: str, language: str) -> str:
    """Fallback for transcripts beyond the single-call limit. Splits, then
    translates and concatenates each piece independently — not a map-reduce
    combine step, since translation shouldn't summarize or drop content."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=6000, chunk_overlap=0)
    parts = splitter.split_text(raw_transcript)
    translated_parts = [_translate_transcript_single_call(part, language) for part in parts]
    return " ".join(translated_parts)


def translate_transcript(raw_transcript: str, language: str = "english") -> str:
    """Translate/normalize the raw transcript into the chosen transcript
    language. This produces the 'Translated Transcript' tab content, and
    (after this step) feeds summarize()/extract_*() (03/04)."""
    if not raw_transcript.strip():
        return ""

    if len(raw_transcript) <= TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT:
        return _translate_transcript_single_call(raw_transcript, language)

    print(
        f"translate_transcript: transcript ({len(raw_transcript):,} chars) exceeds the "
        f"single-call limit ({TRANSCRIPT_SINGLE_CALL_CHAR_LIMIT:,} chars) — "
        "falling back to chunked translation."
    )
    return _translate_transcript_chunked(raw_transcript, language)


## Routing + top-level entry point

`transcribe_all()` is what `09_agents.ipynb` calls. It now takes
**`audio_type`** (`"auto"` or `"hinglish"`) — which ASR path to use — and
returns the RAW transcript ONLY (a plain string). Translation is a separate
call the caller makes afterward with whatever **transcript language** it
wants, independent of `audio_type`.

In [ ]:
def transcribe_chunk_raw(chunk_path: str, audio_type: str = "auto") -> str:
    """Route one chunk to the right RAW transcription path.
    - "auto"     -> Whisper, auto-detected language (see transcribe_chunk_whisper_raw)
    - "hinglish" -> Hinglish-specialized ASR model, raw mixed-script output
    """
    if audio_type.lower() == "hinglish":
        return transcribe_chunk_hinglish_raw(chunk_path)
    return transcribe_chunk_whisper_raw(chunk_path)


def transcribe_all(chunks: list, audio_type: str = "auto") -> str:
    """Transcribe every chunk and return the RAW transcript (exactly what was
    said, auto-detected language/script, never forced). Translation into a
    chosen output language is a separate step — see translate_transcript()
    above, called explicitly by content_agent in 09_agents.ipynb."""
    raw_transcript = ""
    engine = "Hinglish ASR (raw, 30s windows)" if audio_type.lower() == "hinglish" else "Whisper (auto-detect, raw)"
    print(f"Using {engine} for transcription.")

    for i, chunk in enumerate(chunks):
        print(f"Transcribing chunk {i + 1}/{len(chunks)}...")
        text = transcribe_chunk_raw(chunk, audio_type=audio_type)
        raw_transcript += text + " "

    raw_transcript = raw_transcript.strip()
    print(f"Raw transcription complete ({len(raw_transcript)} characters).")
    return raw_transcript
